In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt

# SMO Algorithm Implementation

In [4]:
class SpiderMonkeyOptimization:
    def __init__(self, n_monkeys, max_iter, search_space, fitness_func):
        self.n_monkeys = n_monkeys            # Population size
        self.max_iter = max_iter              # Maximum iterations
        self.search_space = search_space      # Search space (list of tuples)
        self.fitness_func = fitness_func      # Fitness function
        self.population = self.init_population()

    def init_population(self):
        # Random initialization within the search space
        population = []
        for _ in range(self.n_monkeys):
            individual = [np.random.uniform(low, high) for low, high in self.search_space]
            population.append(individual)
        return np.array(population)

    def get_fitness(self, individual):
        return self.fitness_func(individual)

    def optimize(self):
        best_individual = None
        best_fitness = float('inf')

        for iteration in range(self.max_iter):
            print(f"Iteration {iteration + 1}/{self.max_iter}")

            # Evaluate the fitness of each monkey in the population
            fitness_values = np.array([self.get_fitness(ind) for ind in self.population])

            # Find the best monkey in the population
            min_fitness_idx = np.argmin(fitness_values)
            if fitness_values[min_fitness_idx] < best_fitness:
                best_fitness = fitness_values[min_fitness_idx]
                best_individual = self.population[min_fitness_idx].copy()

            # Update positions (this is simplified for the demo)
            for i in range(self.n_monkeys):
                if i != min_fitness_idx:
                    self.population[i] = self.population[i] + np.random.uniform(-1, 1) * (best_individual - self.population[i])

            print(f"Best fitness so far: {best_fitness}")

        return best_individual, best_fitness


# Load Dataset

In [5]:
file_path = 'dataset_wellnessdimensions.csv'  # Update this to your correct file path
data = pd.read_csv(file_path)

In [6]:
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [7]:

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/vedantagarwal/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/vedantagarwal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/vedantagarwal/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [8]:
def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and word not in punctuation]
    return ' '.join(cleaned_tokens)

In [9]:
# Apply preprocessing
data['Cleaned_Text'] = data['Text'].apply(preprocess_text)
data['Cleaned_Explanations'] = data['Explanations'].apply(preprocess_text)

# TF-IDF feature extraction
tfidf_vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_text = tfidf_vectorizer.fit_transform(data['Cleaned_Text'])
tfidf_explanations = tfidf_vectorizer.fit_transform(data['Cleaned_Explanations'])

# Combine the features
combined_data = np.hstack([tfidf_text.toarray(), tfidf_explanations.toarray()])

# Split data into training and validation sets
X_train, X_val = train_test_split(combined_data, test_size=0.2, random_state=42)

In [22]:
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam  # Import Adam optimizer

def build_autoencoder(n_neurons, dropout_rate, l2_reg, learning_rate):
    input_dim = X_train.shape[1]
    
    # Encoder
    input_layer = layers.Input(shape=(input_dim,))
    encoded = layers.Dense(n_neurons, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(input_layer)
    encoded = layers.Dropout(dropout_rate)(encoded)
    encoded = layers.Reshape((n_neurons, 1))(encoded)
    
    # Attention Layer
    def attention_3d_block(inputs):
        attention = layers.Dense(inputs.shape[-1], activation='tanh')(inputs)
        attention = layers.Dense(1, activation='softmax')(attention)
        attention = layers.Lambda(lambda x: tf.squeeze(x, -1))(attention)
        attention = layers.Lambda(lambda x: tf.expand_dims(x, -1))(attention)
        weighted_inputs = inputs * attention
        return weighted_inputs

    attention = attention_3d_block(encoded)
    
    # Flatten and Decoder
    flattened = layers.Flatten()(attention)
    decoded = layers.Dense(n_neurons, activation='relu')(flattened)
    decoded = layers.Dropout(dropout_rate)(decoded)
    decoded = layers.Dense(input_dim, activation='sigmoid')(decoded)
    
    # Autoencoder model
    autoencoder = models.Model(input_layer, decoded)
    autoencoder.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')  # Use Adam directly
    
    print("Autoencoder Model Summary:")
    autoencoder.summary()  # Print model structure to check if it's correctly defined
    
    return autoencoder


In [23]:
def train_autoencoder(n_neurons, dropout_rate, l2_reg, learning_rate):
    autoencoder = build_autoencoder(n_neurons, dropout_rate, l2_reg, learning_rate)
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)
    
    history = autoencoder.fit(X_train, X_train, 
                              epochs=50, 
                              batch_size=32, 
                              validation_data=(X_val, X_val), 
                              callbacks=[early_stopping, reduce_lr], 
                              verbose=0)
    
    return history


In [24]:
print("Training before optimization...")
history_before = train_autoencoder(n_neurons=256, dropout_rate=0.3, l2_reg=0.001, learning_rate=0.001)

Training before optimization...
Autoencoder Model Summary:
Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_5 (InputLayer)        [(None, 2000)]               0         []                            
                                                                                                  
 dense_12 (Dense)            (None, 256)                  512256    ['input_5[0][0]']             
                                                                                                  
 dropout_5 (Dropout)         (None, 256)                  0         ['dense_12[0][0]']            
                                                                                                  
 reshape_4 (Reshape)         (None, 256, 1)               0         ['dropout_5[0][0]']           
                                 

In [26]:
# 2. Optimize using SMO (Define search space and SMO implementation)
search_space = [
    (64, 512),    # Number of neurons in encoder
    (0.1, 0.5),   # Dropout rate
    (0.0001, 0.01),  # L2 regularization
    (0.0001, 0.01)   # Learning rate
]

# Define the objective function for SMO optimization
def objective_function(params):
    # Unpack parameters
    n_neurons = int(params[0])
    dropout_rate = params[1]
    l2_reg = params[2]
    learning_rate = params[3]
    
    # Train and return validation loss
    history = train_autoencoder(n_neurons, dropout_rate, l2_reg, learning_rate)
    val_loss = min(history.history['val_loss'])
    return val_loss

# SMO parameters
smo = SpiderMonkeyOptimization(
    n_monkeys=20,  # Population size
    max_iter=20,   # Maximum iterations
    search_space=search_space,  # Search space
    fitness_func=objective_function  # Objective function
)


In [27]:
# Run SMO to find the best hyperparameters
print("Running SMO optimization...")
best_params, best_loss = smo.optimize()

# 3. Training AFTER optimization using best hyperparameters
print(f"Best Parameters found: {best_params}")
history_after = train_autoencoder(n_neurons=int(best_params[0]), dropout_rate=best_params[1], 
                                  l2_reg=best_params[2], learning_rate=best_params[3])


Running SMO optimization...
Iteration 1/20
Autoencoder Model Summary:
Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_6 (InputLayer)        [(None, 2000)]               0         []                            
                                                                                                  
 dense_17 (Dense)            (None, 321)                  642321    ['input_6[0][0]']             
                                                                                                  
 dropout_7 (Dropout)         (None, 321)                  0         ['dense_17[0][0]']            
                                                                                                  
 reshape_5 (Reshape)         (None, 321, 1)               0         ['dropout_7[0][0]']           
                      

Autoencoder Model Summary:
Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_7 (InputLayer)        [(None, 2000)]               0         []                            
                                                                                                  
 dense_22 (Dense)            (None, 148)                  296148    ['input_7[0][0]']             
                                                                                                  
 dropout_9 (Dropout)         (None, 148)                  0         ['dense_22[0][0]']            
                                                                                                  
 reshape_6 (Reshape)         (None, 148, 1)               0         ['dropout_9[0][0]']           
                                                                 

Autoencoder Model Summary:
Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_8 (InputLayer)        [(None, 2000)]               0         []                            
                                                                                                  
 dense_27 (Dense)            (None, 199)                  398199    ['input_8[0][0]']             
                                                                                                  
 dropout_11 (Dropout)        (None, 199)                  0         ['dense_27[0][0]']            
                                                                                                  
 reshape_7 (Reshape)         (None, 199, 1)               0         ['dropout_11[0][0]']          
                                                                 

Autoencoder Model Summary:
Model: "model_5"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_9 (InputLayer)        [(None, 2000)]               0         []                            
                                                                                                  
 dense_32 (Dense)            (None, 197)                  394197    ['input_9[0][0]']             
                                                                                                  
 dropout_13 (Dropout)        (None, 197)                  0         ['dense_32[0][0]']            
                                                                                                  
 reshape_8 (Reshape)         (None, 197, 1)               0         ['dropout_13[0][0]']          
                                                                 

Autoencoder Model Summary:
Model: "model_6"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_10 (InputLayer)       [(None, 2000)]               0         []                            
                                                                                                  
 dense_37 (Dense)            (None, 472)                  944472    ['input_10[0][0]']            
                                                                                                  
 dropout_15 (Dropout)        (None, 472)                  0         ['dense_37[0][0]']            
                                                                                                  
 reshape_9 (Reshape)         (None, 472, 1)               0         ['dropout_15[0][0]']          
                                                                 

...

Autoencoder Model Summary:
Model: "model_40"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_44 (InputLayer)       [(None, 2000)]               0         []                            
                                                                                                  
 dense_207 (Dense)           (None, 477)                  954477    ['input_44[0][0]']            
                                                                                                  
 dropout_83 (Dropout)        (None, 477)                  0         ['dense_207[0][0]']           
                                                                                                  
 reshape_43 (Reshape)        (None, 477, 1)               0         ['dropout_83[0][0]']          
                                                                

Autoencoder Model Summary:
Model: "model_41"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_45 (InputLayer)       [(None, 2000)]               0         []                            
                                                                                                  
 dense_212 (Dense)           (None, 308)                  616308    ['input_45[0][0]']            
                                                                                                  
 dropout_85 (Dropout)        (None, 308)                  0         ['dense_212[0][0]']           
                                                                                                  
 reshape_44 (Reshape)        (None, 308, 1)               0         ['dropout_85[0][0]']          
                                                                

Best fitness so far: -1299442.625
Iteration 3/20
Autoencoder Model Summary:
Model: "model_42"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_46 (InputLayer)       [(None, 2000)]               0         []                            
                                                                                                  
 dense_217 (Dense)           (None, 178)                  356178    ['input_46[0][0]']            
                                                                                                  
 dropout_87 (Dropout)        (None, 178)                  0         ['dense_217[0][0]']           
                                                                                                  
 reshape_45 (Reshape)        (None, 178, 1)               0         ['dropout_87[0][0]']          
               

Autoencoder Model Summary:
Model: "model_43"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_47 (InputLayer)       [(None, 2000)]               0         []                            
                                                                                                  
 dense_222 (Dense)           (None, 104)                  208104    ['input_47[0][0]']            
                                                                                                  
 dropout_89 (Dropout)        (None, 104)                  0         ['dense_222[0][0]']           
                                                                                                  
 reshape_46 (Reshape)        (None, 104, 1)               0         ['dropout_89[0][0]']          
                                                                

ValueError: Invalid value -0.34087811772806664 received for `rate`, expected a value between 0 and 1.

In [ ]:
# 4. Compare Losses

# Plot training and validation loss before and after optimization
plt.figure(figsize=(10, 6))

# Loss before optimization
plt.plot(history_before.history['loss'], label='Train Loss Before Optimization')
plt.plot(history_before.history['val_loss'], label='Validation Loss Before Optimization')

# Loss after optimization
plt.plot(history_after.history['loss'], label='Train Loss After Optimization')
plt.plot(history_after.history['val_loss'], label='Validation Loss After Optimization')

plt.title('Training and Validation Loss Before and After Optimization')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()